In [5]:
%%duckdb


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,455,1 Ave & E 44 St,40.750020,-73.969053,265,Stanton St & Chrystie St,40.722293,-73.991475,18660,Subscriber,1960.0,2,2015-01-01 00:01:00,2015-01-01 00:24:00,23
1,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
2,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6


## Problem 1: Trip duration

### Part 1: Build a Regression Model

Build a regression to predict trip duration by using
- Day of time
- Distance between start and end stations (there might be more than one way to measure it)
- Hour of day
- Weekend indicator
- Don't forget to model bias (this one is intentionally not used in lecture)
- Also any thing you want to end

### Part 2: Experiment Design

- Ensure that you properly design your experiment to report unbiased performance metric you choose

### Part 3 [Optional]: Visualize

- Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
- Estimate trip duration for those say 10 trips
- Visualize them on map using `pydeck` by using redish color for slower trips and greener for faster trips.

In [ ]:
%%duckdb -o trips

with dataset as (
  select 
    * exclude (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as start_at,
    strptime(stoptime,  ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as stop_at
  from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
  using sample 200000
),
features as (
  select
    date_diff('minute', start_at, stop_at) as duration_min,
    st_distance_sphere(
      st_point("start station longitude", "start station latitude"),
      st_point("end station longitude", "end station latitude")
    ) / 1000.0 as distance_km,
    hour(start_at) as hour_of_day,
    isodow(start_at) as day_of_week,
    case when isodow(start_at) in (6,7) then 1 else 0 end as is_weekend,
    gender,
    usertype,
    2016 - "birth year" as age,
    random() as rnd
  from dataset
)
select * from features
where duration_min between 1 and 180
  and distance_km > 0
  and age is not null
  and gender is not null

In [ ]:
import numpy as np
import pandas as pd

feature_cols = ["distance_km", "hour_of_day", "day_of_week", "is_weekend", "age", "gender", "usertype"]
dummy_cols = ["gender", "usertype"]

X_df = pd.get_dummies(trips[feature_cols], columns=dummy_cols, drop_first=True)
X = np.column_stack([np.ones(len(X_df)), X_df.values.astype(float)])  # ilk sütun = bias
y = trips["duration_min"].values.astype(float)
feature_names = ["bias"] + list(X_df.columns)

is_test = trips["rnd"].values < 0.2
X_train, y_train = X[~is_test], y[~is_test]
X_test,  y_test  = X[is_test],  y[is_test]

beta, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)

for name, coef in zip(feature_names, beta):
    print(f"{name:15s} {coef: .4f}")

In [ ]:
y_pred_test = X_test @ beta

mae = np.median(np.abs(y_pred_test - y_test))
rmse = np.sqrt(np.mean((y_pred_test - y_test) ** 2))

baseline_pred = np.full_like(y_test, y_train.mean())
baseline_rmse = np.sqrt(np.mean((baseline_pred - y_test) ** 2))

print(f"Test MAE  : {mae:.2f} dakika")
print(f"Test RMSE : {rmse:.2f} dakika")
print(f"Baseline RMSE (sadece ortalama): {baseline_rmse:.2f} dakika")

In [ ]:
%%duckdb -o stations

select distinct
  "start station id" as station_id,
  "start station name" as name,
  "start station latitude" as lat,
  "start station longitude" as lon
from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
using sample 5000

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

rng = np.random.default_rng(42)
idx_start = rng.integers(0, len(stations), 10)
idx_end = rng.integers(0, len(stations), 10)

fake_trips = pd.DataFrame({
    "start_lat": stations.loc[idx_start, "lat"].values,
    "start_lon": stations.loc[idx_start, "lon"].values,
    "end_lat": stations.loc[idx_end, "lat"].values,
    "end_lon": stations.loc[idx_end, "lon"].values,
})
fake_trips["distance_km"] = haversine_km(
    fake_trips.start_lat, fake_trips.start_lon, fake_trips.end_lat, fake_trips.end_lon
)
fake_trips["hour_of_day"] = 12
fake_trips["day_of_week"] = 3
fake_trips["is_weekend"] = 0
fake_trips["age"] = 30
fake_trips["gender"] = 1
fake_trips["usertype"] = "Subscriber"

X_fake_df = pd.get_dummies(fake_trips[feature_cols], columns=dummy_cols, drop_first=True)
X_fake_df = X_fake_df.reindex(columns=X_df.columns, fill_value=0)
X_fake = np.column_stack([np.ones(len(X_fake_df)), X_fake_df.values.astype(float)])

fake_trips["pred_duration_min"] = X_fake @ beta
fake_trips["pred_speed_kmh"] = fake_trips["distance_km"] / (fake_trips["pred_duration_min"] / 60)
fake_trips

In [ ]:
import pydeck as pdk

speed = fake_trips["pred_speed_kmh"]
norm = (speed - speed.min()) / (speed.max() - speed.min() + 1e-9)  # 0=yavaş, 1=hızlı

fake_trips["color"] = [[int(255 * (1 - n)), int(255 * n), 40, 200] for n in norm]
fake_trips["path"] = fake_trips.apply(
    lambda r: [[r.start_lon, r.start_lat], [r.end_lon, r.end_lat]], axis=1
)

layer = pdk.Layer(
    "PathLayer",
    data=fake_trips,
    get_path="path",
    get_color="color",
    width_min_pixels=4,
    pickable=True,
)

view_state = pdk.ViewState(
    longitude=fake_trips.start_lon.mean(),
    latitude=fake_trips.start_lat.mean(),
    zoom=11,
)

pdk.Deck(layers=[layer], initial_view_state=view_state, map_provider="carto", map_style="dark").show()

### Problem 2: Extending Naive Bayesian

### Part 1: Expand the NB Regression Idea to continous variable

$$
P(gender = a, speed_{bike} = x) = P(gender = a) P(speed_{bike} = x | gender = a)
$$

- Note that $P(speed_{bike} = x | gender = a)$ is  continous distribution.
- Expand the idea
- Build a predictive model for estimation biker gender using the bike speed ?

### Part 2: Use Visualization to decide best distribution 

- How should be $P(speed_{bike} = x | gender = a)$ modeled

In [1]:
%%duckdb -o gender_1_duration


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
) where gender =1 
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
1,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6
2,384,Fulton St & Waverly Ave,40.683178,-73.965964,399,Lafayette Ave & St James Pl,40.688515,-73.964763,19610,Subscriber,1969.0,1,2015-01-01 00:04:00,2015-01-01 00:07:00,3


In [ ]:
%%duckdb -o gender_speed

with dataset as (
  select 
    * exclude (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as start_at,
    strptime(stoptime,  ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as stop_at
  from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
  using sample 200000
),
features as (
  select
    gender,
    date_diff('second', start_at, stop_at) / 3600.0 as duration_hour,
    st_distance_sphere(
      st_point("start station longitude", "start station latitude"),
      st_point("end station longitude", "end station latitude")
    ) / 1000.0 as distance_km,
    random() as rnd
  from dataset
)
select *, distance_km / duration_hour as speed_kmh
from features
where duration_hour > 0
  and gender in (1, 2)
  and distance_km / duration_hour between 1 and 40

In [ ]:
import numpy as np

train = gender_speed[gender_speed.rnd >= 0.2]
test = gender_speed[gender_speed.rnd < 0.2]

stats = train.groupby("gender")["speed_kmh"].agg(["mean", "std"])
print(stats)

def gaussian_pdf(x, mu, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Not: gerçek sınıf oranlarını (priors) kullanırsak %76-%24 dengesizliği önseli ezer ve
# model her zaman çoğunluk sınıfını (gender=1) tahmin eder (balanced accuracy = 0.5,
# yani hiç ayırt etmiyor). Hızın gerçekte taşıdığı zayıf ama gerçek sinyali görmek için
# eşit önsel kullanıyoruz; bu Bayes kuralının kasıtlı bir varyasyonu.
equal_priors = {g: 1 / len(stats) for g in stats.index}

def predict_gender(speed):
    posteriors = {g: equal_priors[g] * gaussian_pdf(speed, stats.loc[g, "mean"], stats.loc[g, "std"]) for g in stats.index}
    return max(posteriors, key=posteriors.get)

test = test.copy()
test["pred_gender"] = test["speed_kmh"].apply(predict_gender)
accuracy = (test["pred_gender"] == test["gender"]).mean()
print(f"Doğruluk (accuracy): {accuracy:.3f}")

majority_baseline = (test["gender"] == train["gender"].mode()[0]).mean()
print(f"Çoğunluk sınıfı baseline: {majority_baseline:.3f}")

# Sınıf dengesizliği accuracy'i yanıltıcı yapabilir, her sınıf için ayrı recall'a bakalım
recall_per_class = (test["pred_gender"] == test["gender"]).groupby(test["gender"]).mean()
balanced_accuracy = recall_per_class.mean()
print(f"Sınıf başına recall:\n{recall_per_class}")
print(f"Balanced accuracy: {balanced_accuracy:.3f}")

In [ ]:
gender_speed["log_speed"] = np.log(gender_speed["speed_kmh"])
train = gender_speed[gender_speed.rnd >= 0.2]
test = gender_speed[gender_speed.rnd < 0.2]

def fit_gaussian(df, col):
    return df.groupby("gender")[col].agg(["mean", "std"])

def evaluate(df, stats_, col, use_equal_priors):
    if use_equal_priors:
        priors_ = {g: 1 / len(stats_) for g in stats_.index}
    else:
        priors_ = train["gender"].value_counts(normalize=True).to_dict()

    def predict(x):
        posteriors = {g: priors_[g] * gaussian_pdf(x, stats_.loc[g, "mean"], stats_.loc[g, "std"]) for g in stats_.index}
        return max(posteriors, key=posteriors.get)

    pred = df[col].apply(predict)
    acc = (pred == df["gender"]).mean()
    recall = (pred == df["gender"]).groupby(df["gender"]).mean()
    return acc, recall.mean(), recall

stats_speed = fit_gaussian(train, "speed_kmh")
stats_log = fit_gaussian(train, "log_speed")

for name, stats_, col, equal in [
    ("ham hız, gerçek prior", stats_speed, "speed_kmh", False),
    ("ham hız, eşit prior", stats_speed, "speed_kmh", True),
    ("log(hız), gerçek prior", stats_log, "log_speed", False),
    ("log(hız), eşit prior", stats_log, "log_speed", True),
]:
    acc, bal_acc, recall = evaluate(test, stats_, col, equal)
    print(f"{name:25s} -> accuracy={acc:.3f}  balanced_accuracy={bal_acc:.3f}  recall={recall.to_dict()}")

In [ ]:
import plotly.express as px

fig = px.histogram(
    gender_speed, x="speed_kmh", color="gender",
    histnorm="probability density", barmode="overlay", nbins=60,
    title="Cinsiyete göre hız dağılımı"
)
fig.show()

gender_speed["log_speed"] = np.log(gender_speed["speed_kmh"])
fig2 = px.histogram(
    gender_speed, x="log_speed", color="gender",
    histnorm="probability density", barmode="overlay", nbins=60,
    title="log(hız) dağılımı - cinsiyete göre"
)
fig2.show()